# *SendGrid:* Permite enviar emails a dispositivos mediante su API con alto soporte para integrarse en aplicaciones

In [ ]:
#laboratorio - sistema multiagentes-Agente de ventas
#SENDGRID_API_KEY=

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [2]:
load_dotenv(override=True)

True

In [4]:
#parte 1: Flujo de trabajo del agente
instructions1 = "Eres un agente de ventas que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío profesionales y serios."

instructions2 = "Eres un agente de ventas con sentido del humor y atractivo \
    que trabaja para ComplAI, una empresa que ofrece una herramienta SaaS para \
    garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
    Redactas correos electrónicos en frío ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Eres un agente de ventas muy activo que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío concisos y directos."

In [ ]:
sales_agent1 = Agent(
    name="Agente de Ventas profesional",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Agente de ventas atractivo",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Agente de ventas ocupado",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [ ]:
result = Runner.run_streamed(sales_agent1, input="Escribe un correo electronico de ventas en frio")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

### Ejecutar Agentes en paralelo

In [ ]:
message = "Escribe un correo electronico de ventas en frio"
with trace("Correos electronicos de ventas en frio 01"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for rsult in results]
for output in outputs:
    print(output + "\n\n")

# Agente Selector

In [ ]:
#crear un agente selector
instructions = "Elige el mejor correo electronico de ventas en frio entre las opciones disponibles.\
    Imagina que eres un cliente y elige el que probablemente te responda.\
    No des explicaciones; responde solo con el correo electronico seleccionado."
sales_picker = Agent(name="sales_picker",
                     instructions=instructions,
                     model="gpt-4o-mini")

In [ ]:
message = "Escribe un correo electronico de ventas en frio"
with trace("Seleccion del equipo de ventas_01"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for rsult in results]
    emails = "Emails de ventas en frio: \n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"El mejor email de ventas: \n{best.final_output}")

# Usar Agentes como Herramientas

In [5]:
sales_agent1 = Agent(
    name="Agente de Ventas profesional",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Agente de ventas atractivo",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Agente de ventas ocupado",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [6]:
sales_agent1

Agent(name='Agente de Ventas profesional', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Eres un agente de ventas que trabaja para ComplAI,     una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío profesionales y serios.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [9]:
@function_tool
def send_email(body: str):
    """ Envía un correo electrónico con el cuerpo indicado a todos los clientes potenciales de ventas. """
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email("walbertorh3@gmail.com")  # Cambiar a tu remitente verificado
    to_email = To("walberto.roblero.condor@gmail.com")  # Cambiar a su receptor
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Email de ventas", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [10]:
send_email

FunctionTool(name='send_email', description='Envía un correo electrónico con el cuerpo indicado a todos los clientes potenciales de ventas.', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B35744720>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [11]:
### convertir los agentes a herramientas
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Escribe un email de ventas en frio.")
tool1

FunctionTool(name='sales_agent1', description='Escribe un email de ventas en frio.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B35746AC0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [12]:
description = "Escribe un correo electronico de ventas en frio."
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [14]:
tools = [tool1, tool2, tool3, send_email]
tools

[FunctionTool(name='sales_agent1', description='Escribe un correo electronico de ventas en frio.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B35747F60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Escribe un correo electronico de ventas en frio.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B357B8180>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails

In [15]:
#Crear el Manager de Ventas (Agente Orquestador)
instructions ="Eres gerente de ventas y trabajas para ComplAI. \
Utilizas las herramientas que te proporcionamos para generar correos electrónicos de ventas en frío. \
Nunca generas correos electrónicos de ventas tú mismo; siempre usas las herramientas. \
Pruebas las tres herramientas de sales_agent una vez antes de elegir la mejor. \
Eliges el mejor correo electrónico y usas la herramienta send_email para enviar el mejor correo electrónico (y solo el mejor) al usuario."

sales_manager = Agent(name="Manager de ventas", instructions=instructions, 
                      tools=tools, model="gpt-4o-mini")

message = "Envia un correo electronico de ventas en frio dirigido a 'Estimado director ejecutivo'"

with trace("Manager de ventas"):
    result = await Runner.run(sales_manager, message)

In [ ]:
## uv pip install --upgrade certifi
#import certifi
#import os
#os.environ["SSL_CERT_FILE"] = certifi.where()

# Handoffs vs Tools
### Tools= Un solo agente mantiene el *control*
### Handoffs= El control y contexto trabajado se *transfiere* entre agentes

In [16]:
subject_instructions = "Puedes escribir un asunto para un correo electronico de ventas en frio.\
Se te proporciona un mensaje y necesitas escribir un asunto para un correo electronico que probablemente obtenga respuesta"

html_istructions = "Puedes convertir un cuerpo de correo electronico de texto a un cuerpo de correo electronico HTML. \
    Se te proporciona un cuerpo de correo electronico de texto que puede tener algun markdown y necesitas convertirlo a un cuerpo\
         de correo electronico HTML con un diseño claro, atractivo y que se adecuado para el asunto."

subject_writer = Agent(name="Escritor de asunto de correo electronico",
                       instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Escribe un asunto para un correo electronico de ventas en frio")

html_converter = Agent(name="Conversor de cuerpo de correo electronico HTML",
                       instructions=html_istructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",
                                   tool_description="Convierte un cuerpo de correo electronico de texto a un cuerpo de correo electronico HTML")


In [19]:
#Funcion de nevio de correo HTML
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envía un correo electrónico con el asunto y el cuerpo HTML a todos los cleintes potenciales de ventas."""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email("walbertorh3@gmail.com")  # Cambiar a tu remitente verificado
    to_email = To("walberto.roblero.condor@gmail.com")  # Cambiar a su receptor
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [18]:
tools = [subject_tool, html_tool, send_html_email]

In [20]:
instructions = "Eres un formateador y remitente de correos electronicos.\
    Recibes el cuerpo de un correo electronico para enviarlo.\
    Primero usas la herrameinta subject_writer para escribir un asunto para el correo electronico,\
    luego usas la herramienta html_converter para convertir el cuerpo a HTML.\
    Finalmente, usas la herramienta send_html_email para enviar el correo electronico con el asunto y el cuerpo HTML."

emailer_agent = Agent(
    name = "Email Manager",
    instructions=instructions,
    tools=tools,
    model = "gpt-4o-mini",
    handoff_description="Convierte un email a HTML y lo envia"
)

In [21]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Escribe un correo electronico de ventas en frio.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B35747F60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Escribe un correo electronico de ventas en frio.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000024B357B8180>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=

In [22]:
sales_manager_instructions = "Eres un gerente de ventas que trabaja para ComplAI. Utilizas las herramientas que se te proporcionan para generar correos electrónicos de ventas en frío. \
Nunca generas correos electrónicos de ventas tú mismo; siempre utilizas las herramientas. \
Pruebas las 3 herramientas del agente de ventas al menos una vez antes de elegir la mejor. \
Puedes usar las herramientas múltiples veces si no estás satisfecho con los resultados del primer intento. \
Seleccionas el mejor correo electrónico usando tu propio criterio sobre cuál será más efectivo. \
Después de elegir el correo electrónico, transfieres al agente Email Manager para formatear y enviar el correo."

sales_manager = Agent(
    name = "Manager de ventas",
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini"
)

message = "Envia un correo electronico de ventas en frio dirigido a 'Estimado director ejecutivo'"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)